# 6주차 ② 과적합과 일반화 · 평가지표 — 실습 4~7  〔빈칸본〕

> **2교시 노트북입니다.** 빈칸은 모두 **7곳 · 13줄** 입니다 —
> 실습 4에 2곳(5줄), 실습 5에 1곳(2줄), 실습 6에 1곳(2줄), 실습 7에 3곳(4줄).
>
> 빈칸마다 **힌트**와 **확인**(채우고 나면 이런 값이 나온다)이 붙어 있습니다.
> 다 채운 노트북은 `09_regularization.ipynb` 로 저장해 **제출**합니다.

> **오늘의 질문**: *"그런데 좋아진 게 진짜인가?"*
> 1교시에는 정확도를 올렸습니다. 이번 교시에는 **그 숫자를 믿어도 되는지** 확인하는 법을 배웁니다.

---

## 실습 4 — 과적합 재현·관찰 ★★ (12분)

**이번 교시의 하이라이트.** 과적합은 말로 들으면 안 와닿습니다. **직접 만듭니다.**

```
   [과소적합]              [적당]                  [과적합]
   공부를 덜 했다          원리를 이해했다          기출문제 답만 외웠다

   훈련 60% / 검증 58%     훈련 90% / 검증 88%      훈련 99% / 검증 82%  ★
      둘 다 낮다              둘 다 높다              훈련만 높다
```

> **과적합의 진단법은 한 줄입니다** ★ —
> **훈련 손실은 계속 내려가는데 검증 손실이 올라가기 시작하는 지점.**
> 그 순간부터 모델은 **배우는 게 아니라 외우고** 있습니다.

### 왜 검증(validation)이 따로 필요한가

5주차에는 **훈련 / 테스트** 두 개만 썼습니다. 오늘 **검증**이 추가됩니다.

| 분할 | 쓰임 | 몇 번 보나 |
|---|---|---|
| **훈련(train)** | 파라미터를 학습 | 매 epoch |
| **검증(validation)** | **설정을 고른다** (lr, 드롭아웃 비율, epoch 수) | 매 epoch |
| **테스트(test)** | **최종 성능 보고** | 딱 한 번 |

> **테스트로 설정을 고르면 테스트에 과적합됩니다** ★.
> 여러 설정을 테스트로 비교해 가장 좋은 걸 고르면, 그 점수는 이미 *"테스트를 보고 고른 점수"* 라
> 새 데이터에 대한 추정치가 아닙니다. 그래서 **고르는 용도의 데이터를 따로** 둡니다.

### 과적합을 부르는 조건

```
   데이터가 적다  +  모델이 크다  +  오래 학습한다  →   외우기 시작한다
```

5주차 모델(파라미터 23만 개)에 **훈련 데이터를 1,000장으로 줄이면** 금방 외웁니다.

In [ ]:
# 셀 1 — 일부러 데이터를 줄인다
import torch, torch.nn as nn
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms
import matplotlib.pyplot as plt

plt.rcParams["font.family"] = "Malgun Gothic"
plt.rcParams["axes.unicode_minus"] = False

device = "cuda" if torch.cuda.is_available() else "cpu"
tf = transforms.Compose([transforms.ToTensor(), transforms.Normalize((0.2860,), (0.3530,))])
full = datasets.FashionMNIST("data", train=True,  download=True, transform=tf)
test = datasets.FashionMNIST("data", train=False, download=True, transform=tf)

SMALL = 1000                                   # ★ 훈련을 1,000장으로

# ───── 실습 4 · 빈칸 ① (2줄) ─────
# 힌트 : ① 앞에서부터 SMALL 장을 훈련으로 ② 그 뒤 2000장을 검증으로. Subset(full, range(...)) (2줄)
# 확인 : 훈련 1000 장 / 검증 2000 장 으로 출력된다



small_loader = DataLoader(small_train, batch_size=64,  shuffle=True)
val_loader   = DataLoader(val_set,     batch_size=256, shuffle=False)
test_loader  = DataLoader(test,        batch_size=256, shuffle=False)
print("훈련", len(small_train), "장 / 검증", len(val_set), "장 / 테스트", len(test), "장")

In [ ]:
# 셀 2 — 모델 만들기 (드롭아웃·BN 은 실습 5 에서 켠다)
def build(dropout=0.0, bn=False):
    torch.manual_seed(0)
    layers = [nn.Flatten(), nn.Linear(784, 256)]
    if bn: layers.append(nn.BatchNorm1d(256))          # ★ 선형 -> BN -> 활성함수
    layers += [nn.ReLU()]
    if dropout: layers.append(nn.Dropout(dropout))
    layers += [nn.Linear(256, 128)]
    if bn: layers.append(nn.BatchNorm1d(128))
    layers += [nn.ReLU()]
    if dropout: layers.append(nn.Dropout(dropout))
    layers += [nn.Linear(128, 10)]
    return nn.Sequential(*layers).to(device)


print(build(dropout=0.3, bn=True))

In [ ]:
# 셀 3 — 훈련 손실과 검증 손실을 매 epoch 기록한다  ★★
def run(model, epochs=30, lr=0.05):
    opt = torch.optim.SGD(model.parameters(), lr=lr, momentum=0.9)
    lossfn = nn.CrossEntropyLoss()
    tr_hist, va_hist = [], []

    for _ in range(epochs):
        # ── 훈련
        model.train(); run_l = n = 0
        for xb, yb in small_loader:
            xb, yb = xb.to(device), yb.to(device)
            loss = lossfn(model(xb), yb)
            opt.zero_grad(); loss.backward(); opt.step()
            run_l += loss.item() * xb.size(0); n += xb.size(0)
        tr_hist.append(run_l / n)

        # ── 검증 (파라미터를 갱신하지 않는다)
# ───── 실습 4 · 빈칸 ② (3줄) ─────
# 힌트 : ① 평가 모드로 ② 기울기를 계산하지 않는 블록 안에서 ③ 검증 손실을 누적 (3줄)
# 확인 : 검증 곡선이 어느 지점에서 바닥을 찍고 다시 올라간다



                xb, yb = xb.to(device), yb.to(device)
                run_l += lossfn(model(xb), yb).item() * xb.size(0); n += xb.size(0)
        va_hist.append(run_l / n)

    return tr_hist, va_hist


tr, va = run(build())

plt.figure(figsize=(7, 4))
plt.plot(tr, label="훈련 손실")
plt.plot(va, label="검증 손실")
plt.axvline(int(torch.tensor(va).argmin()), color="red", ls="--", label="검증 최저점")
plt.xlabel("epoch"); plt.ylabel("loss"); plt.legend(); plt.grid(alpha=0.3)
plt.title("과적합 — 두 곡선이 갈라지는 순간"); plt.show()

best_ep = int(torch.tensor(va).argmin())
print(f"검증 손실이 가장 낮았던 epoch : {best_ep + 1}")
print(f"마지막 epoch   | 훈련 {tr[-1]:.4f} / 검증 {va[-1]:.4f}")
print(f"검증 최저 시점 | 훈련 {tr[best_ep]:.4f} / 검증 {va[best_ep]:.4f}")

> **관찰 포인트 ★★** — 교수님 PC 실측값입니다.
>
> ```
> 검증 손실이 가장 낮았던 epoch : 7
> 마지막 epoch   | 훈련 0.1244 / 검증 1.2650
> 검증 최저 시점 | 훈련 0.2930 / 검증 0.6602
> ```
>
> **훈련 손실은 0.29 → 0.12 로 계속 내려갑니다.** 그런데 검증 손실은
> **0.66 에서 바닥을 찍고 1.27 까지 다시 올라갔습니다.** 빨간 점선이 그 지점입니다.
>
> **7 epoch 이후의 23 epoch 은 성능을 떨어뜨리는 데 쓴 것입니다.**
> 훈련 손실만 보고 있었다면 *"잘 되고 있네"* 라고 착각했을 구간입니다.
>
> 5주차에는 **테스트 정확도 하나**만 봤습니다.
> 그래서 *"더 오래 학습하면 더 좋아지겠지"* 라고 생각하기 쉬웠죠. **아닙니다.**
> **두 곡선을 함께 보는 것**이 오늘 배우는 가장 중요한 습관입니다.

> **조기 종료(early stopping)** — 위 그림의 **빨간 점선에서 멈추는 것**입니다.
> 구현은 *"검증 손실이 N epoch 동안 안 좋아지면 중단"* 이 전부이고,
> 실무에서는 그 시점의 가중치를 `best_model.pt` 로 저장해 둡니다(3교시 실습 10).

---

## 실습 5 — 드롭아웃과 배치 정규화를 적용해 재비교 ★ (10분)

### 드롭아웃 — 일부러 뉴런을 끈다

```
   [훈련 시]  매 배치마다 무작위로 p 비율의 뉴런을 0 으로

      ○ ○ ○ ○ ○          ○ ✗ ○ ✗ ○        ✗ ○ ○ ○ ✗
      배치 1              배치 2            배치 3
      -> 특정 뉴런에만 의존할 수 없게 된다  ->  여러 경로로 풀도록 강제

   [추론 시]  전부 켠다. 대신 출력 크기를 맞춘다   <- model.eval() 이 이걸 한다 ★
```

### 배치 정규화 — 층 입력을 매번 정규화

```
   5주차: 입력 데이터를 한 번 정규화했다  (transforms.Normalize)
   오늘:  각 층의 입력을 매 배치마다 정규화한다  (BatchNorm)

        Linear -> BatchNorm -> ReLU        ★ 이 순서
```

| 효과 | 설명 |
|---|---|
| 학습이 빨라진다 | 각 층이 안정된 분포를 받는다 → 더 큰 lr 을 써도 된다 |
| 일반화에도 도움 | 배치마다 통계가 조금씩 달라 잡음 역할을 한다 |
| **훈련/추론 동작이 다르다** ★ | 훈련은 배치 통계, 추론은 **누적 평균**. `eval()` 필수 |

> **드롭아웃과 배치 정규화 둘 다 `train()`/`eval()` 로 동작이 갈립니다.**
> 이것이 5주차부터 모드 전환 습관을 들인 이유입니다.

> **가중치 감쇠(weight decay)** — 큰 가중치에 벌점을 주어 모델을 단순하게 유지합니다.
> `optim.SGD(..., weight_decay=1e-4)` 한 줄이면 됩니다. 사용법만 알고 넘어갑니다.

In [ ]:
# 셀 4 — 세 조건 비교
설정 = {
    "기본":           dict(dropout=0.0, bn=False),

# ───── 실습 5 · 빈칸 ① (2줄) ─────
# 힌트 : ① 드롭아웃 0.3 만 ② 드롭아웃 0.3 + 배치정규화 (2줄)
# 확인 : 검증 곡선 3개가 그려지고, 드롭아웃 쪽 훈련 손실이 더 높게 나온다


}

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
for name, kw in 설정.items():
    tr, va = run(build(**kw))
    ax[0].plot(tr, label=name)
    ax[1].plot(va, label=name)
    print(f"{name:14s} | 훈련 {tr[-1]:.4f} | 검증 최저 {min(va):.4f} "
          f"(epoch {va.index(min(va))+1})")

ax[0].set_title("훈련 손실"); ax[1].set_title("검증 손실")
for a in ax: a.set_xlabel("epoch"); a.legend(); a.grid(alpha=0.3)
plt.tight_layout(); plt.show()

> **관찰 포인트 ★** — 실측값입니다. **오른쪽(검증) 열을 보세요.**
>
> ```
> 기본          | 훈련 0.1244 | 검증 최저 0.6602 (epoch 7)
> 드롭아웃 0.3   | 훈련 0.2517 | 검증 최저 0.6137 (epoch 6)
> 드롭아웃 + BN  | 훈련 0.0872 | 검증 최저 0.5923 (epoch 6)
> ```
>
> **드롭아웃만 넣으면 훈련 손실이 0.1244 → 0.2517 로 오히려 높아집니다.**
> 뉴런을 꺼 놓고 학습하니 당연합니다. 그런데 **검증 최저점은 0.6602 → 0.6137 로 낮아졌습니다.**
>
> *"훈련 성적을 일부러 깎아서 실전 성적을 올린다"* — **이것이 일반화 기법의 본질입니다.**
>
> **BN 을 함께 넣은 세 번째 줄은 훈련 손실이 다시 0.0872 로 낮습니다.**
> 배치 정규화가 학습을 가속하기 때문입니다. 그래도 **검증 최저점은 0.5923 으로 셋 중 가장 좋습니다** —
> 우리가 보는 것은 **훈련 손실이 아니라 검증 손실**이라는 점을 여기서 한 번 더 확인하세요.

---

## 실습 6 — `eval()` 을 빼면 어떻게 되나 ★ (5분)

**5주차에 `model.eval()` 을 "일단 쓰라"고 한 이유가 오늘 밝혀집니다.**
드롭아웃은 **훈련할 때와 추론할 때 동작이 다릅니다.**

In [ ]:
# 셀 5 — eval() 유무 비교
model = build(dropout=0.3)
run(model, epochs=10)

def acc(m, use_eval):
# ───── 실습 6 · 빈칸 ① (2줄) ─────
# 힌트 : use_eval 이 True 면 평가 모드, False 면 (일부러) 훈련 모드로 둔다 (2줄)
# 확인 : train() 쪽 정확도가 낮고, 두 번 부르면 값이 서로 다르다


    c = t = 0
    with torch.no_grad():
        for xb, yb in test_loader:
            xb, yb = xb.to(device), yb.to(device)
            c += (m(xb).argmax(1) == yb).sum().item(); t += yb.size(0)
    return c / t


print(f"model.eval()  -> {acc(model, True)*100:.2f}%")
print(f"model.eval()  -> {acc(model, True)*100:.2f}%   <- 두 번 해도 같습니다")
print(f"model.train() -> {acc(model, False)*100:.2f}%   <- 낮고, 실행할 때마다 달라진다")
print(f"model.train() -> {acc(model, False)*100:.2f}%   <- 보세요, 값이 또 다릅니다")

> **결과 해석 ★★** — 실측값입니다.
>
> ```
> model.eval()  -> 76.07%
> model.eval()  -> 76.07%   <- 두 번 해도 같습니다
> model.train() -> 73.73%   <- 낮고, 실행할 때마다 달라진다
> model.train() -> 73.64%   <- 보세요, 값이 또 다릅니다
> ```
>
> `train()` 모드로 평가하면 **정확도가 2%p 넘게 낮고, 두 번 실행하면 값이 다릅니다**(73.73 / 73.64).
> 드롭아웃이 매번 다른 뉴런을 끄기 때문입니다.
> *(정확도가 76% 대인 것은 훈련 데이터가 1,000장뿐이라 그렇습니다 — 5주차의 86% 와 비교하지 마세요.)*
>
> **`model.eval()` 을 빼먹는 것은 학기 내내 가장 흔한 버그**입니다. 오늘 한 번 겪어 두세요.
> 배치 정규화도 같은 이유로 `eval()` 이 필요합니다 — 훈련은 배치 통계, 추론은 누적 평균을 씁니다.

---

## 실습 7 — 혼동행렬·정밀도·재현율·F1 (10분)

정밀도·재현율·F1 의 **정의는 이미 배웠습니다**(빅데이터분석프로그래밍 13·14주).
여기서는 **"PyTorch 결과를 sklearn 에 어떻게 넘기는가"** 와 **혼동행렬을 읽는 법**만 합니다.

In [ ]:
# 셀 6 — 테스트셋 전체 예측을 모은다
LABELS = ["티셔츠","바지","풀오버","드레스","코트","샌들","셔츠","스니커즈","가방","앵클부츠"]

model.eval()
all_pred, all_true = [], []
with torch.no_grad():
    for xb, yb in test_loader:
        xb = xb.to(device)

# ───── 실습 7 · 빈칸 ① (2줄) ─────
# 힌트 : 예측 클래스 번호를 CPU 로 옮겨 담고, 정답도 담는다. .cpu() 를 빼면 sklearn 에서 오류 (2줄)
# 확인 : (10000,) 으로 출력된다



y_pred = torch.cat(all_pred).numpy()
y_true = torch.cat(all_true).numpy()
print("예측 수집 완료 :", y_pred.shape)

> **포인트**: `.cpu()` 를 빼면 sklearn 에 넘길 때 오류가 납니다 — 3주차 실습 8의 그 이야기입니다.
> GPU 텐서는 NumPy 로 바로 못 바꿉니다.

In [ ]:
# 셀 7 — 지표와 혼동행렬
# ───── 실습 7 · 빈칸 ② (1줄) ─────
# 힌트 : sklearn.metrics 에서 classification_report 와 confusion_matrix 를 가져온다
# 확인 : 10개 클래스의 precision/recall/f1 표가 출력된다


print(classification_report(y_true, y_pred, target_names=LABELS, digits=3))

# ───── 실습 7 · 빈칸 ③ (1줄) ─────
# 힌트 : 정답과 예측으로 혼동행렬을 만든다
# 확인 : (10, 10) 행렬


fig, ax = plt.subplots(figsize=(7, 6))
im = ax.imshow(cm, cmap="Blues")
ax.set_xticks(range(10)); ax.set_xticklabels(LABELS, rotation=90)
ax.set_yticks(range(10)); ax.set_yticklabels(LABELS)
ax.set_xlabel("예측"); ax.set_ylabel("정답"); ax.set_title("혼동행렬")
for i in range(10):
    for j in range(10):
        if cm[i, j] > 0:
            ax.text(j, i, cm[i, j], ha="center", va="center", fontsize=7,
                    color="white" if cm[i, j] > cm.max()/2 else "black")
plt.colorbar(im); plt.tight_layout(); plt.show()

In [ ]:
# 셀 8 — 대각선 밖에서 가장 큰 혼동 5쌍
import numpy as np

off = cm.copy()
np.fill_diagonal(off, 0)
pairs = np.dstack(np.unravel_index(np.argsort(off, axis=None)[::-1], off.shape))[0][:5]

print("가장 자주 헷갈리는 조합")
for i, j in pairs:
    print(f"  정답 {LABELS[i]:8s} -> 예측 {LABELS[j]:8s} : {off[i, j]}건")

> **관찰 포인트 ★★**: 대각선이 정답입니다. **대각선 밖의 큰 숫자를 찾으세요.** (실측값)
>
> ```
>   정답 풀오버  -> 예측 코트   : 320건
>   정답 티셔츠  -> 예측 셔츠   : 206건
>   정답 셔츠   -> 예측 코트   : 193건
>   정답 드레스  -> 예측 코트   : 167건
>   정답 풀오버  -> 예측 셔츠   : 162건
> ```
>
> 상위 5개가 전부 **셔츠 ↔ 티셔츠 ↔ 코트 ↔ 풀오버** 구역입니다.
> 5주차 실습 11에서 **이미지로** 봤던 그것을 오늘 **숫자로** 확인하는 것입니다.
>
> `classification_report` 의 **셔츠 행**도 함께 보세요 — precision 0.477 / recall 0.559 로
> 10개 클래스 중 가장 낮습니다. **바지(0.947)·가방(0.899)** 과 비교하면 차이가 분명합니다.

### 정확도만 보면 안 되는 경우

```
   예: 100건 중 양성 2건인 불량품 검사

     "전부 정상"이라고만 답하는 모델  ->  정확도 98%  ★ 그런데 쓸모가 0
                                          재현율 0%
```

| 지표 | 언제 중요한가 |
|---|---|
| **정밀도(precision)** | 잘못된 경보의 비용이 클 때 (스팸함에 중요 메일이 가면 안 됨) |
| **재현율(recall)** | 놓치는 비용이 클 때 (암 진단에서 환자를 놓치면 안 됨) |
| **F1** | 둘의 균형 |

> **핵심 ★**: FashionMNIST 는 클래스가 고르게 있어서 정확도로 충분합니다.
> 그런데 **현실 데이터는 대개 불균형**합니다. *"정확도 98%"* 를 보면
> **먼저 클래스 비율을 확인하는 습관**을 가지세요. 중간고사 출제 지점입니다.

---

## 2교시 핵심 정리

| 개념 | 한 줄 정의 |
|------|-----------|
| **과적합 진단** ★★ | 훈련 손실↓ + **검증 손실↑** 인 지점부터 |
| **세 분할** | 훈련(학습) / 검증(**설정 선택**) / 테스트(**최종 1회**) |
| **조기 종료** | 검증 최저점에서 멈춘다. 그 시점 가중치를 저장 |
| **드롭아웃** | 훈련 시 무작위로 뉴런을 끈다. **추론 시엔 전부 켠다** → `eval()` 필수 ★ |
| **배치 정규화** | 층 입력을 매 배치 정규화. **선형 → BN → 활성함수** 순서 |
| **`model.eval()`** ★★ | 빼먹으면 정확도가 낮고 **실행할 때마다 달라진다** |
| **가중치 감쇠** | `weight_decay=1e-4` 한 줄. 큰 가중치에 벌점 |
| **정확도의 한계** | 클래스 불균형이면 무의미. 정밀도·재현율·F1·혼동행렬 |

### 반드시 외우기 ★

```python
nn.Dropout(0.3)
nn.BatchNorm1d(256)            # 선형 → BN → ReLU 순서

model.train()   # 학습 시
model.eval()    # 평가 시 ★ 빼먹으면 드롭아웃/BN 이 잘못 동작
```

### 2교시 체크리스트

- [ ] 과적합의 진단법을 **한 줄로** 말할 수 있다 ★★
- [ ] 검증이 테스트와 **별도로** 필요한 이유를 안다
- [ ] 데이터를 1,000장으로 줄여 **두 곡선이 갈라지는 것**을 직접 봤다 ★
- [ ] 드롭아웃이 훈련 손실을 **높이는데도** 쓰는 이유를 안다
- [ ] `model.eval()` 을 빼면 정확도가 낮고 매번 달라지는 것을 확인했다 ★★
- [ ] BN 의 위치(선형 → BN → 활성함수)를 안다
- [ ] `classification_report` 와 혼동행렬을 출력했다
- [ ] 정확도만 보면 안 되는 경우를 예로 들 수 있다

### 저장

**출력이 살아 있는 상태로** `09_regularization.ipynb` 로 저장합니다. **과제 제출 대상입니다.**

> **다음 시간 예고**: 오늘 실험을 여러 번 실행했습니다. 그런데 **다음 주에 다시 실행하면
> 같은 결과가 나올까요?** 시드를 안 고정했다면 안 나옵니다.
> 3교시에는 **실험을 통제하는 법**을 배웁니다 — 시드 고정, 모델 저장,
> **끊긴 학습 이어서 하기**, 그리고 실험 5개를 한 화면에서 비교하는 TensorBoard 입니다.